In [0]:
import sys
sys.path.append("/Workspace/Shared/databricks-practice/utils/")

import pyspark.sql.functions as F

from utils.functions import *

In [0]:
df = spark.read.table("databricks_practice.bronze.bronze_table")
display(df.select("country_code", "currency").distinct())

In [0]:
# Country_code, Currency tem nulos

In [0]:
bronze_table = "databricks_practice.bronze.bronze_table"
silver_table = "databricks_practice.silver.dim_mercado"
partition_col = "dt_ingestion_partition"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.silver.dim_mercado (
  location_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  country_code STRING,
  currency STRING,
  dt_ingestion_partition DATE,
  update_ts TIMESTAMP
) USING DELTA;




In [0]:
df_new = load_bronze_new_partitions(spark, bronze_table, silver_table, partition_col)

In [0]:
df_mercado_silver = clean_mercado(df_new)

In [0]:
df_dim_mercado = (df_mercado_silver) \
    .select("country_code", "currency", "dt_ingestion_partition") \
    .withColumn("dt_ingestion_partition", F.col("dt_ingestion_partition").cast("date")) \
    .dropDuplicates() \
    .dropna() 
  #  .withColumn("market_sk", F.sha2(F.concat_ws("||", F.col("country_code")), 256)

In [0]:
df_dim_mercado.createOrReplaceTempView("df_dim_mercado")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricks_practice.silver

In [0]:
# Teste Upsert ou Overwrite

In [0]:
%sql
MERGE INTO databricks_practice.silver.dim_mercado AS silver
USING df_dim_mercado AS bronze
ON silver.country_code = bronze.country_code AND silver.currency = bronze.currency

WHEN MATCHED THEN UPDATE SET
  silver.country_code = bronze.country_code,
  silver.currency = bronze.currency,
  silver.dt_ingestion_partition = bronze.dt_ingestion_partition,
  silver.update_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  country_code, currency, dt_ingestion_partition, update_ts
)
VALUES (
  bronze.country_code, bronze.currency, bronze.dt_ingestion_partition, current_timestamp()
);

In [0]:
%sql
SELECT * FROM databricks_practice.silver.dim_mercado

In [0]:
display(spark.table("databricks_practice.silver.dim_mercado").groupBy("country_code").count())